# Experiment 2: Double-Pipe Conjugate Heat Transfer

**Aim:** Analyse heat exchange between hot and cold laminar streams separated by a conducting wall.

**Numerical method:** Steady finite-volume convection-diffusion in the fluid zones coupled to conduction in the solid wall.

> **Scope:** The geometry is a two-dimensional stacked-domain analogue. The velocity profiles are prescribed to isolate conjugate thermal transport.

This Google Colab notebook is self-contained. It reconstructs the tested solver inside the temporary Colab runtime, runs it, displays quantitative plots and an inline field contour, and packages every output file for download.

> **Colab workflow:** Choose a run mode, use **Runtime > Run all**, then download the results ZIP from the final cell. `quick` is only a workflow check; use `lab` or an approved `custom` case for submission.

In [ ]:
#@title 1. Prepare the self-contained Colab runtime

from pathlib import Path
import shutil
import subprocess
import sys
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

IN_COLAB = False
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    colab_files = None


def run_solver(script_path, arguments, output_dir, timeout=1800):
    output_dir = Path(output_dir)
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    command = [sys.executable, str(script_path), *[str(value) for value in arguments]]
    print("Running:", " ".join(command))
    completed = subprocess.run(command, cwd=output_dir, capture_output=True, text=True, timeout=timeout)
    (output_dir / "solver_stdout.log").write_text(completed.stdout, encoding="utf-8")
    (output_dir / "solver_stderr.log").write_text(completed.stderr, encoding="utf-8")
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.returncode != 0:
        raise RuntimeError(completed.stderr.strip() or f"Solver exited with code {completed.returncode}")
    if completed.stderr.strip():
        print(completed.stderr.strip())
    return output_dir


def read_csv(path):
    data = np.genfromtxt(path, delimiter=",", names=True, dtype=None, encoding="utf-8")
    return np.atleast_1d(data)


def show_summary(path):
    text = Path(path).read_text(encoding="utf-8").strip()
    display(Markdown(f"### Solver summary\n```text\n{text}\n```"))
    return text


def list_outputs(output_dir):
    rows = []
    for path in sorted(Path(output_dir).iterdir()):
        if path.is_file():
            size = path.stat().st_size
            if size < 1024:
                size_text = f"{size} B"
            elif size < 1024**2:
                size_text = f"{size / 1024:.1f} KB"
            else:
                size_text = f"{size / 1024**2:.1f} MB"
            rows.append((path.name, size_text))
    table = "\n".join(f"| `{name}` | {size} |" for name, size in rows)
    display(Markdown(f"### Generated files\n| File | Size |\n|---|---:|\n{table}"))
    return rows


def read_structured_vtk(path):
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    dims_index = next(i for i, line in enumerate(lines) if line.startswith("DIMENSIONS "))
    _, nx, ny, nz = lines[dims_index].split()
    nx, ny, nz = int(nx), int(ny), int(nz)
    if nz != 1:
        raise ValueError("Only 2D structured VTK files are supported by this notebook preview")
    points_index = next(i for i, line in enumerate(lines) if line.startswith("POINTS "))
    npoints = int(lines[points_index].split()[1])
    points = np.array([[float(v) for v in lines[points_index + 1 + k].split()[:3]] for k in range(npoints)])
    x = points[:, 0].reshape(ny, nx)
    y = points[:, 1].reshape(ny, nx)
    data_index = next(i for i, line in enumerate(lines) if line.startswith("POINT_DATA ")) + 1
    scalars = {}
    vectors = {}
    i = data_index
    while i < len(lines):
        if lines[i].startswith("SCALARS "):
            name = lines[i].split()[1]
            i += 2
            values = np.array([float(lines[i + k].split()[0]) for k in range(npoints)])
            scalars[name] = values.reshape(ny, nx)
            i += npoints
        elif lines[i].startswith("VECTORS "):
            name = lines[i].split()[1]
            i += 1
            values = np.array([[float(v) for v in lines[i + k].split()[:3]] for k in range(npoints)])
            vectors[name] = (values[:, 0].reshape(ny, nx), values[:, 1].reshape(ny, nx))
            i += npoints
        else:
            i += 1
    return x, y, scalars, vectors


def plot_vtk_scalar(path, preferred_field=None):
    path = Path(path)
    if not path.is_file():
        print(f"VTK preview skipped: {path.name} was not generated")
        return
    x, y, scalars, _ = read_structured_vtk(path)
    if not scalars:
        print("VTK preview skipped: no scalar field found")
        return
    field = preferred_field if preferred_field in scalars else next(iter(scalars))
    plt.figure(figsize=(8, 4.8))
    image = plt.contourf(x, y, scalars[field], levels=30)
    plt.colorbar(image, label=field.replace("_", " "))
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"{field.replace('_', ' ').title()} - inline field preview")
    plt.axis("equal")
    plt.tight_layout()
    plt.show()


def make_results_zip(output_dir, experiment_number):
    output_dir = Path(output_dir)
    archive = output_dir.parent / f"experiment_{experiment_number:02d}_results.zip"
    if archive.exists():
        archive.unlink()
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as handle:
        for path in sorted(output_dir.rglob("*")):
            if path.is_file():
                handle.write(path, path.relative_to(output_dir))
    print(f"Results archive: {archive} ({archive.stat().st_size / 1024:.1f} KB)")
    return archive


EXPERIMENT = 2
ROOT = Path("/content/cfd_lab_exp02") if IN_COLAB else Path.cwd() / "_colab_runtime_exp02"
CODE_DIR = ROOT / "code"
OUTPUT_DIR = ROOT / "outputs"
CODE_DIR.mkdir(parents=True, exist_ok=True)
(CODE_DIR / "common.py").write_text('from __future__ import annotations\n\nimport csv\nimport math\nfrom pathlib import Path\nfrom typing import Iterable, Mapping\n\nimport numpy as np\n\n\ndef trapezoidal_integral(values: np.ndarray, coordinates: np.ndarray) -> float:\n    y = np.asarray(values, dtype=float).ravel()\n    x = np.asarray(coordinates, dtype=float).ravel()\n    if y.size != x.size or y.size < 2:\n        raise ValueError("trapezoidal integration requires equally sized arrays with at least two points")\n    return float(np.sum(0.5 * (y[1:] + y[:-1]) * np.diff(x)))\n\n\ndef write_csv(path: str | Path, header: Iterable[str], rows: Iterable[Iterable[object]]) -> None:\n    with Path(path).open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(list(header))\n        writer.writerows(rows)\n\n\ndef mesh_xy(nx: int, ny: int, lx: float = 1.0, ly: float = 1.0, x0: float = 0.0, y0: float = 0.0) -> tuple[np.ndarray, np.ndarray]:\n    x = np.linspace(x0, x0 + lx, nx)\n    y = np.linspace(y0, y0 + ly, ny)\n    return np.meshgrid(x, y)\n\n\ndef write_structured_grid_vtk(\n    path: str | Path,\n    x: np.ndarray,\n    y: np.ndarray,\n    scalars: Mapping[str, np.ndarray] | None = None,\n    vectors: Mapping[str, tuple[np.ndarray, np.ndarray]] | None = None,\n    title: str = "CFD lab output",\n) -> None:\n    x = np.asarray(x, dtype=float)\n    y = np.asarray(y, dtype=float)\n    if x.shape != y.shape or x.ndim != 2:\n        raise ValueError("x and y must be equally shaped 2D arrays")\n    ny, nx = x.shape\n    scalar_map = dict(scalars or {})\n    vector_map = dict(vectors or {})\n    for name, values in scalar_map.items():\n        if np.asarray(values).shape != x.shape:\n            raise ValueError(f"scalar {name} has shape {np.asarray(values).shape}, expected {x.shape}")\n    for name, values in vector_map.items():\n        if len(values) != 2 or np.asarray(values[0]).shape != x.shape or np.asarray(values[1]).shape != x.shape:\n            raise ValueError(f"vector {name} has incompatible shape")\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET STRUCTURED_GRID\\n")\n        handle.write(f"DIMENSIONS {nx} {ny} 1\\n")\n        handle.write(f"POINTS {nx * ny} float\\n")\n        for j in range(ny):\n            for i in range(nx):\n                handle.write(f"{x[j, i]:.12e} {y[j, i]:.12e} 0\\n")\n        handle.write(f"POINT_DATA {nx * ny}\\n")\n        for name, values in scalar_map.items():\n            data = np.asarray(values, dtype=float)\n            handle.write(f"SCALARS {name} float 1\\n")\n            handle.write("LOOKUP_TABLE default\\n")\n            for value in data.ravel(order="C"):\n                handle.write(f"{value:.12e}\\n")\n        for name, values in vector_map.items():\n            vx = np.asarray(values[0], dtype=float)\n            vy = np.asarray(values[1], dtype=float)\n            handle.write(f"VECTORS {name} float\\n")\n            for ax, ay in zip(vx.ravel(order="C"), vy.ravel(order="C")):\n                handle.write(f"{ax:.12e} {ay:.12e} 0\\n")\n\n\ndef write_polyline_vtk(path: str | Path, x: np.ndarray, y: np.ndarray, title: str = "polyline", close: bool = False) -> None:\n    x = np.asarray(x, dtype=float).ravel()\n    y = np.asarray(y, dtype=float).ravel()\n    if x.size != y.size or x.size < 2:\n        raise ValueError("polyline requires equally sized x and y arrays")\n    ids = list(range(x.size))\n    if close:\n        ids.append(0)\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {x.size} float\\n")\n        for xi, yi in zip(x, y):\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"LINES 1 {len(ids) + 1}\\n")\n        handle.write(f"{len(ids)} {\' \'.join(str(i) for i in ids)}\\n")\n\n\ndef write_multiline_vtk(path: str | Path, lines: list[tuple[np.ndarray, np.ndarray]], title: str = "lines") -> None:\n    points: list[tuple[float, float]] = []\n    connectivity: list[list[int]] = []\n    for x, y in lines:\n        xa = np.asarray(x, dtype=float).ravel()\n        ya = np.asarray(y, dtype=float).ravel()\n        if xa.size != ya.size or xa.size < 2:\n            raise ValueError("each line needs equally sized x and y arrays")\n        ids = list(range(len(points), len(points) + xa.size))\n        points.extend(zip(xa.tolist(), ya.tolist()))\n        connectivity.append(ids)\n    size = sum(len(ids) + 1 for ids in connectivity)\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {len(points)} float\\n")\n        for xi, yi in points:\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"LINES {len(connectivity)} {size}\\n")\n        for ids in connectivity:\n            handle.write(f"{len(ids)} {\' \'.join(str(i) for i in ids)}\\n")\n\n\ndef write_points_vtk(path: str | Path, x: np.ndarray, y: np.ndarray, scalars: Mapping[str, np.ndarray] | None = None, title: str = "points") -> None:\n    xa = np.asarray(x, dtype=float).ravel()\n    ya = np.asarray(y, dtype=float).ravel()\n    if xa.size != ya.size:\n        raise ValueError("point arrays must have equal size")\n    scalar_map = dict(scalars or {})\n    with Path(path).open("w", encoding="utf-8") as handle:\n        handle.write("# vtk DataFile Version 3.0\\n")\n        handle.write(f"{title}\\n")\n        handle.write("ASCII\\n")\n        handle.write("DATASET POLYDATA\\n")\n        handle.write(f"POINTS {xa.size} float\\n")\n        for xi, yi in zip(xa, ya):\n            handle.write(f"{xi:.12e} {yi:.12e} 0\\n")\n        handle.write(f"VERTICES {xa.size} {2 * xa.size}\\n")\n        for i in range(xa.size):\n            handle.write(f"1 {i}\\n")\n        if scalar_map:\n            handle.write(f"POINT_DATA {xa.size}\\n")\n            for name, values in scalar_map.items():\n                data = np.asarray(values, dtype=float).ravel()\n                if data.size != xa.size:\n                    raise ValueError(f"scalar {name} has wrong size")\n                handle.write(f"SCALARS {name} float 1\\n")\n                handle.write("LOOKUP_TABLE default\\n")\n                for value in data:\n                    handle.write(f"{value:.12e}\\n")\n\n\ndef gradient(field: np.ndarray, dx: float, dy: float) -> tuple[np.ndarray, np.ndarray]:\n    gy, gx = np.gradient(field, dy, dx, edge_order=2)\n    return gx, gy\n\n\ndef laplacian(field: np.ndarray, dx: float, dy: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    out[1:-1, 1:-1] = (\n        (field[1:-1, 2:] - 2.0 * field[1:-1, 1:-1] + field[1:-1, :-2]) / (dx * dx)\n        + (field[2:, 1:-1] - 2.0 * field[1:-1, 1:-1] + field[:-2, 1:-1]) / (dy * dy)\n    )\n    return out\n\n\ndef upwind_derivative_x(field: np.ndarray, velocity: np.ndarray, dx: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    backward = (field[1:-1, 1:-1] - field[1:-1, :-2]) / dx\n    forward = (field[1:-1, 2:] - field[1:-1, 1:-1]) / dx\n    out[1:-1, 1:-1] = np.where(velocity[1:-1, 1:-1] >= 0.0, backward, forward)\n    return out\n\n\ndef upwind_derivative_y(field: np.ndarray, velocity: np.ndarray, dy: float) -> np.ndarray:\n    out = np.zeros_like(field)\n    backward = (field[1:-1, 1:-1] - field[:-2, 1:-1]) / dy\n    forward = (field[2:, 1:-1] - field[1:-1, 1:-1]) / dy\n    out[1:-1, 1:-1] = np.where(velocity[1:-1, 1:-1] >= 0.0, backward, forward)\n    return out\n\n\ndef poisson_jacobi(phi: np.ndarray, rhs: np.ndarray, dx: float, dy: float, iterations: int, boundary: str = "neumann", mask: np.ndarray | None = None) -> np.ndarray:\n    dx2 = dx * dx\n    dy2 = dy * dy\n    denom = 2.0 * (dx2 + dy2)\n    for _ in range(iterations):\n        old = phi.copy()\n        phi[1:-1, 1:-1] = (\n            (old[1:-1, 2:] + old[1:-1, :-2]) * dy2\n            + (old[2:, 1:-1] + old[:-2, 1:-1]) * dx2\n            - rhs[1:-1, 1:-1] * dx2 * dy2\n        ) / denom\n        if mask is not None:\n            phi[mask > 0.5] = 0.0\n        if boundary == "neumann":\n            phi[:, 0] = phi[:, 1]\n            phi[:, -1] = 0.0\n            phi[0, :] = phi[1, :]\n            phi[-1, :] = phi[-2, :]\n        elif boundary == "closed":\n            phi[:, 0] = phi[:, 1]\n            phi[:, -1] = phi[:, -2]\n            phi[0, :] = phi[1, :]\n            phi[-1, :] = phi[-2, :]\n            phi -= np.mean(phi)\n        else:\n            raise ValueError("unsupported pressure boundary")\n    return phi\n\n\ndef tdma(a: np.ndarray, b: np.ndarray, c: np.ndarray, d: np.ndarray) -> np.ndarray:\n    a = np.asarray(a, dtype=float).copy()\n    b = np.asarray(b, dtype=float).copy()\n    c = np.asarray(c, dtype=float).copy()\n    d = np.asarray(d, dtype=float).copy()\n    n = d.size\n    if n == 0 or a.size != n or b.size != n or c.size != n:\n        raise ValueError("TDMA arrays must have equal nonzero size")\n    for i in range(1, n):\n        if abs(b[i - 1]) < 1e-30:\n            raise FloatingPointError("zero TDMA pivot")\n        factor = a[i] / b[i - 1]\n        b[i] -= factor * c[i - 1]\n        d[i] -= factor * d[i - 1]\n    x = np.empty(n, dtype=float)\n    if abs(b[-1]) < 1e-30:\n        raise FloatingPointError("zero TDMA pivot")\n    x[-1] = d[-1] / b[-1]\n    for i in range(n - 2, -1, -1):\n        if abs(b[i]) < 1e-30:\n            raise FloatingPointError("zero TDMA pivot")\n        x[i] = (d[i] - c[i] * x[i + 1]) / b[i]\n    return x\n\n\ndef relative_l2(numerical: np.ndarray, reference: np.ndarray) -> float:\n    a = np.asarray(numerical, dtype=float)\n    b = np.asarray(reference, dtype=float)\n    denominator = float(np.linalg.norm(b.ravel()))\n    numerator = float(np.linalg.norm((a - b).ravel()))\n    return numerator / max(denominator, 1e-30)\n\n\ndef require_finite(*arrays: np.ndarray) -> None:\n    for array in arrays:\n        if not np.all(np.isfinite(array)):\n            raise FloatingPointError("non-finite numerical field detected")\n\n\ndef positive_float(value: str, name: str) -> float:\n    result = float(value)\n    if not math.isfinite(result) or result <= 0.0:\n        raise ValueError(f"{name} must be positive and finite")\n    return result\n\n\ndef positive_int(value: str, name: str, minimum: int = 1) -> int:\n    result = int(value)\n    if result < minimum:\n        raise ValueError(f"{name} must be at least {minimum}")\n    return result\n', encoding="utf-8")
(CODE_DIR / 'exp02_double_pipe_cht_fvm.py').write_text('from __future__ import annotations\n\nimport csv\nimport sys\n\nimport numpy as np\n\nfrom common import require_finite, trapezoidal_integral, write_structured_grid_vtk\n\n\ndef main() -> int:\n    nx = int(sys.argv[1]) if len(sys.argv) > 1 else 160\n    ny_cold = int(sys.argv[2]) if len(sys.argv) > 2 else 20\n    ny_wall = int(sys.argv[3]) if len(sys.argv) > 3 else 6\n    ny_hot = int(sys.argv[4]) if len(sys.argv) > 4 else 20\n    re_hot = float(sys.argv[5]) if len(sys.argv) > 5 else 100.0\n    re_cold = float(sys.argv[6]) if len(sys.argv) > 6 else 100.0\n    ks_ratio = float(sys.argv[7]) if len(sys.argv) > 7 else 20.0\n    counter_flow = int(sys.argv[8]) if len(sys.argv) > 8 else 1\n    max_iterations = int(sys.argv[9]) if len(sys.argv) > 9 else 3000\n    prandtl = float(sys.argv[10]) if len(sys.argv) > 10 else 7.0\n    t_hot_in = float(sys.argv[11]) if len(sys.argv) > 11 else 1.0\n    t_cold_in = float(sys.argv[12]) if len(sys.argv) > 12 else 0.0\n    u_hot_mean = float(sys.argv[13]) if len(sys.argv) > 13 else 1.0\n    u_cold_mean = float(sys.argv[14]) if len(sys.argv) > 14 else 1.0\n    if nx < 20 or ny_cold < 4 or ny_wall < 2 or ny_hot < 4:\n        raise ValueError("grid is too small")\n    if re_hot <= 0.0 or re_cold <= 0.0 or ks_ratio <= 0.0 or prandtl <= 0.0 or max_iterations < 1:\n        raise ValueError("physical and iteration parameters must be positive")\n    counter_flow = 1 if counter_flow else 0\n    ny = ny_cold + ny_wall + ny_hot\n    length = 8.0\n    height = 2.0\n    x1 = np.linspace(0.0, length, nx)\n    y1 = np.linspace(0.0, height, ny)\n    x, y = np.meshgrid(x1, y1)\n    dx = x1[1] - x1[0]\n    dy = y1[1] - y1[0]\n    zone = np.zeros((ny, nx), dtype=float)\n    zone[ny_cold:ny_cold + ny_wall, :] = 1.0\n    zone[ny_cold + ny_wall:, :] = 2.0\n    velocity = np.zeros((ny, nx), dtype=float)\n    cold_eta = np.linspace(0.0, 1.0, ny_cold)\n    hot_eta = np.linspace(0.0, 1.0, ny_hot)\n    cold_profile = 6.0 * cold_eta * (1.0 - cold_eta)\n    hot_profile = 6.0 * hot_eta * (1.0 - hot_eta)\n    cold_direction = -1.0 if counter_flow else 1.0\n    velocity[:ny_cold, :] = cold_direction * u_cold_mean * cold_profile[:, None]\n    velocity[ny_cold + ny_wall:, :] = u_hot_mean * hot_profile[:, None]\n    gamma_hot = max(abs(u_hot_mean), 1e-8) / (re_hot * prandtl)\n    gamma_cold = max(abs(u_cold_mean), 1e-8) / (re_cold * prandtl)\n    gamma_wall = ks_ratio * 0.5 * (gamma_hot + gamma_cold)\n    gamma = np.full((ny, nx), gamma_cold, dtype=float)\n    gamma[ny_cold:ny_cold + ny_wall, :] = gamma_wall\n    gamma[ny_cold + ny_wall:, :] = gamma_hot\n    temperature = np.full((ny, nx), 0.5 * (t_hot_in + t_cold_in), dtype=float)\n    temperature[:ny_cold, :] = t_cold_in\n    temperature[ny_cold + ny_wall:, :] = t_hot_in\n    temperature[ny_cold:ny_cold + ny_wall, :] = np.linspace(t_cold_in, t_hot_in, ny_wall)[:, None]\n    relax = 0.82\n    history: list[list[float]] = []\n\n    def apply_boundaries(field: np.ndarray) -> None:\n        field[ny_cold + ny_wall:, 0] = t_hot_in\n        field[ny_cold + ny_wall:, -1] = field[ny_cold + ny_wall:, -2]\n        if counter_flow:\n            field[:ny_cold, -1] = t_cold_in\n            field[:ny_cold, 0] = field[:ny_cold, 1]\n        else:\n            field[:ny_cold, 0] = t_cold_in\n            field[:ny_cold, -1] = field[:ny_cold, -2]\n        field[ny_cold:ny_cold + ny_wall, 0] = field[ny_cold:ny_cold + ny_wall, 1]\n        field[ny_cold:ny_cold + ny_wall, -1] = field[ny_cold:ny_cold + ny_wall, -2]\n        field[0, :] = field[1, :]\n        field[-1, :] = field[-2, :]\n\n    apply_boundaries(temperature)\n    for iteration in range(1, max_iterations + 1):\n        old = temperature.copy()\n        gp = gamma[1:-1, 1:-1]\n        ge = 2.0 * gp * gamma[1:-1, 2:] / np.maximum(gp + gamma[1:-1, 2:], 1e-30)\n        gw = 2.0 * gp * gamma[1:-1, :-2] / np.maximum(gp + gamma[1:-1, :-2], 1e-30)\n        gn = 2.0 * gp * gamma[2:, 1:-1] / np.maximum(gp + gamma[2:, 1:-1], 1e-30)\n        gs = 2.0 * gp * gamma[:-2, 1:-1] / np.maximum(gp + gamma[:-2, 1:-1], 1e-30)\n        flow = velocity[1:-1, 1:-1] * dy\n        a_e = ge * dy / dx + np.maximum(-flow, 0.0)\n        a_w = gw * dy / dx + np.maximum(flow, 0.0)\n        a_n = gn * dx / dy\n        a_s = gs * dx / dy\n        a_p = np.maximum(a_e + a_w + a_n + a_s, 1e-30)\n        candidate = (\n            a_e * old[1:-1, 2:]\n            + a_w * old[1:-1, :-2]\n            + a_n * old[2:, 1:-1]\n            + a_s * old[:-2, 1:-1]\n        ) / a_p\n        temperature[1:-1, 1:-1] = (1.0 - relax) * old[1:-1, 1:-1] + relax * candidate\n        apply_boundaries(temperature)\n        residual = float(np.max(np.abs(temperature - old)))\n        hot_out = float(np.average(temperature[ny_cold + ny_wall:, -1], weights=np.maximum(velocity[ny_cold + ny_wall:, -1], 0.0) + 1e-12))\n        cold_column = 0 if counter_flow else -1\n        cold_weights = np.abs(velocity[:ny_cold, cold_column]) + 1e-12\n        cold_out = float(np.average(temperature[:ny_cold, cold_column], weights=cold_weights))\n        history.append([iteration, residual, hot_out, cold_out, float(np.min(temperature)), float(np.max(temperature))])\n        require_finite(temperature)\n        if iteration == 1 or iteration == max_iterations or iteration % max(1, max_iterations // 10) == 0:\n            print(f"iteration={iteration} residual={residual:.6e} hot_out={hot_out:.6f} cold_out={cold_out:.6f}")\n        if residual < 1e-9 and iteration > 50:\n            break\n    hot_in_weights = np.maximum(velocity[ny_cold + ny_wall:, 0], 0.0) + 1e-12\n    hot_out_weights = np.maximum(velocity[ny_cold + ny_wall:, -1], 0.0) + 1e-12\n    cold_in_col = -1 if counter_flow else 0\n    cold_out_col = 0 if counter_flow else -1\n    cold_in_weights = np.abs(velocity[:ny_cold, cold_in_col]) + 1e-12\n    cold_out_weights = np.abs(velocity[:ny_cold, cold_out_col]) + 1e-12\n    hot_in = float(np.average(temperature[ny_cold + ny_wall:, 0], weights=hot_in_weights))\n    hot_out = float(np.average(temperature[ny_cold + ny_wall:, -1], weights=hot_out_weights))\n    cold_in = float(np.average(temperature[:ny_cold, cold_in_col], weights=cold_in_weights))\n    cold_out = float(np.average(temperature[:ny_cold, cold_out_col], weights=cold_out_weights))\n    mdot_hot = trapezoidal_integral(np.maximum(velocity[ny_cold + ny_wall:, 0], 0.0), y1[ny_cold + ny_wall:])\n    mdot_cold = trapezoidal_integral(np.abs(velocity[:ny_cold, cold_in_col]), y1[:ny_cold])\n    q_hot = mdot_hot * (hot_in - hot_out)\n    q_cold = mdot_cold * (cold_out - cold_in)\n    imbalance = abs(q_hot - q_cold) / max(abs(q_hot), abs(q_cold), 1e-30)\n    effectiveness = q_cold / max(min(mdot_hot, mdot_cold) * (t_hot_in - t_cold_in), 1e-30)\n    write_structured_grid_vtk("double_pipe_cht.vtk", x, y, {"temperature": temperature, "zone": zone, "conductivity_ratio_field": gamma / max(gamma_cold, 1e-30)}, {"velocity": (velocity, np.zeros_like(velocity))}, "Double-pipe conjugate heat transfer")\n    with open("history.csv", "w", newline="", encoding="utf-8") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["iteration", "residual_linf", "hot_outlet_temperature", "cold_outlet_temperature", "minimum_temperature", "maximum_temperature"])\n        writer.writerows(history)\n    with open("summary.txt", "w", encoding="utf-8") as handle:\n        handle.write("Teaching-level 2D conjugate heat-transfer analogue\\n")\n        handle.write(f"nx {nx}\\nny_cold {ny_cold}\\nny_wall {ny_wall}\\nny_hot {ny_hot}\\n")\n        handle.write(f"counter_flow {counter_flow}\\nRe_hot {re_hot:.12g}\\nRe_cold {re_cold:.12g}\\nPr {prandtl:.12g}\\nks_by_kf {ks_ratio:.12g}\\n")\n        handle.write(f"hot_inlet_temperature {hot_in:.12e}\\nhot_outlet_temperature {hot_out:.12e}\\n")\n        handle.write(f"cold_inlet_temperature {cold_in:.12e}\\ncold_outlet_temperature {cold_out:.12e}\\n")\n        handle.write(f"hot_stream_heat_loss {q_hot:.12e}\\ncold_stream_heat_gain {q_cold:.12e}\\n")\n        handle.write(f"relative_energy_imbalance {imbalance:.12e}\\neffectiveness {effectiveness:.12e}\\n")\n        handle.write(f"iterations_completed {len(history)}\\nfinal_residual_linf {history[-1][1]:.12e}\\n")\n    print("Wrote double_pipe_cht.vtk, history.csv, summary.txt")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', encoding="utf-8")
SCRIPT = CODE_DIR / 'exp02_double_pipe_cht_fvm.py'
print("Runtime:", "Google Colab" if IN_COLAB else "local notebook test")
print("Working folder:", ROOT)
print("No package installation or external file upload is required.")


## Parameters

Use `quick` to test the workflow, `lab` for the prescribed case, and `custom` only when changing the form values for the assignment.

In [ ]:
#@title 2. Select run mode and parameters
RUN_MODE = "quick" #@param ["quick", "lab", "custom"]

nx = 160 #@param {type:"integer"}
ny_cold = 20 #@param {type:"integer"}
ny_wall = 6 #@param {type:"integer"}
ny_hot = 20 #@param {type:"integer"}
Re_hot = 100.0 #@param {type:"number"}
Re_cold = 100.0 #@param {type:"number"}
ks_by_kf = 20.0 #@param {type:"number"}
counter_flow = True #@param {type:"boolean"}
max_iterations = 3000 #@param {type:"integer"}
Pr = 7.0 #@param {type:"number"}
T_hot_in = 1.0 #@param {type:"number"}
T_cold_in = 0.0 #@param {type:"number"}
U_hot = 1.0 #@param {type:"number"}
U_cold = 1.0 #@param {type:"number"}

QUICK_ARGS = [40, 6, 2, 6, 100, 100, 20, 1, 20, 7, 1, 0, 1, 1]
LAB_ARGS = [160, 20, 6, 20, 100, 100, 20, 1, 3000, 7, 1, 0, 1, 1]
CUSTOM_ARGS = [nx, ny_cold, ny_wall, ny_hot, Re_hot, Re_cold, ks_by_kf, int(counter_flow), max_iterations, Pr, T_hot_in, T_cold_in, U_hot, U_cold]
ARGS = QUICK_ARGS if RUN_MODE == "quick" else LAB_ARGS if RUN_MODE == "lab" else CUSTOM_ARGS
print("Run mode:", RUN_MODE)
print("Arguments:", ARGS)


## Run the solver

In [ ]:
#@title 3. Run the numerical solver
OUT = run_solver(SCRIPT, ARGS, OUTPUT_DIR)


## Numerical summary

In [ ]:
#@title 4. Read the solver summary
summary_text = show_summary(OUT / 'summary.txt')


## Quantitative plots

In [ ]:
#@title 5. Plot the primary numerical results
history = read_csv(OUT / "history.csv")
plt.figure(figsize=(7, 4))
plt.semilogy(history["iteration"], history["residual_linf"])
plt.xlabel("Iteration")
plt.ylabel("L-infinity residual")
plt.title("Temperature convergence")
plt.grid(True)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history["iteration"], history["hot_outlet_temperature"], label="Hot outlet")
plt.plot(history["iteration"], history["cold_outlet_temperature"], label="Cold outlet")
plt.xlabel("Iteration")
plt.ylabel("Temperature")
plt.title("Outlet-temperature convergence")
plt.legend()
plt.grid(True)
plt.show()

## Inline field visualization

This preview removes the need for ParaView during the basic laboratory workflow. The VTK file is still included in the downloaded results for optional advanced post-processing.

In [ ]:
#@title 6. Display the final field and generated files
plot_vtk_scalar(OUT / 'double_pipe_cht.vtk', 'temperature')
files = list_outputs(OUT)


## Download results

In [ ]:
#@title 7. Package and download the results
DOWNLOAD_RESULTS = False #@param {type:"boolean"}
RESULTS_ZIP = make_results_zip(OUT, 2)
if DOWNLOAD_RESULTS:
    if IN_COLAB:
        colab_files.download(str(RESULTS_ZIP))
    else:
        print("Local test mode. Archive retained at:", RESULTS_ZIP)
else:
    print("Set DOWNLOAD_RESULTS to True and run this cell when ready to download.")


## 7. Compulsory assignment

Run $Re=50,100,200$ and compare outlet temperatures, heat-transfer rate, effectiveness, and energy imbalance. Repeat one case for conductivity ratios 1 and 100.

## Report checklist

- State the governing equations and boundary conditions.
- Report the complete parameter set and grid.
- Include residual or error history.
- Include the required quantitative comparison.
- Include at least one correctly labelled inline field figure; VTK/ParaView figures are optional unless specifically assigned.
- Distinguish observed numerical behaviour from physical interpretation.
- State the limitations of the supplied teaching model.